In [1]:
# Data Handling
import os
import numpy as np
import pandas as pd

# TensorFlow / Keras
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.utils import load_img, img_to_array

# Model Building
from tensorflow.keras.models import Sequential

# Layers
from tensorflow.keras.layers import (
    Dense,
    Conv2D,
    Dropout,
    Flatten,
    MaxPooling2D
)

In [2]:
TRAIN_DIR = 'images/train'
TEST_DIR = 'images/validation'

In [3]:
def createDataFrame(dir):
    img_path = []
    labels = []

    for label in os.listdir(dir):
        for image_name in os.listdir(os.path.join(dir, label)):
            img_path.append(
                os.path.join(dir, label, image_name)
            )
            labels.append(label)

        print(label, "completed")

    return img_path, labels

In [4]:
train = pd.DataFrame()
train['image'], train['label']  = createDataFrame(TRAIN_DIR)

angry completed
disgust completed
fear completed
happy completed
neutral completed
sad completed
surprise completed


In [5]:
print(train)

                                image     label
0            images/train\angry\0.jpg     angry
1            images/train\angry\1.jpg     angry
2           images/train\angry\10.jpg     angry
3        images/train\angry\10002.jpg     angry
4        images/train\angry\10016.jpg     angry
...                               ...       ...
28816  images/train\surprise\9969.jpg  surprise
28817  images/train\surprise\9985.jpg  surprise
28818  images/train\surprise\9990.jpg  surprise
28819  images/train\surprise\9992.jpg  surprise
28820  images/train\surprise\9996.jpg  surprise

[28821 rows x 2 columns]


In [6]:
test = pd.DataFrame()
test['image'], test['label']  = createDataFrame(TEST_DIR)

angry completed
disgust completed
fear completed
happy completed
neutral completed
sad completed
surprise completed


In [7]:
from tqdm import tqdm

def extract_features(images):
    features = []

    for image in tqdm(images):
        img = load_img(
            image,
            color_mode="grayscale",
            target_size=(48, 48)
        )

        img = np.array(img)
        features.append(img)

    features = np.array(features)
    features = features.reshape(
        len(features),
        48,
        48,
        1
    )

    return features

In [8]:
!pip install pillow


[notice] A new release of pip is available: 24.0 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [36]:
import PIL
from PIL import Image

print(PIL.__version__)
print(Image)

12.2.0
<module 'PIL.Image' from 'c:\\student-project\\emotion-detection\\.venv\\Lib\\site-packages\\PIL\\Image.py'>


In [37]:
import keras
print(keras.__version__)

3.14.1


In [9]:
print(train['image'].head())

0        images/train\angry\0.jpg
1        images/train\angry\1.jpg
2       images/train\angry\10.jpg
3    images/train\angry\10002.jpg
4    images/train\angry\10016.jpg
Name: image, dtype: str


In [39]:
from PIL import Image

img = Image.open(train['image'].iloc[0])
print(img.size)

(48, 48)


In [10]:
from PIL import Image
import numpy as np
from tqdm import tqdm

def extract_features(images):
    features = []

    for image in tqdm(images):
        img = Image.open(image).convert("L")   # grayscale
        img = img.resize((48, 48))

        img = np.array(img)
        features.append(img)

    return np.array(features)

In [11]:
train_features = extract_features(train['image'])

100%|████████████████████████████████████████████████████████| 28821/28821 [01:45<00:00, 273.67it/s]


In [12]:
test_features = extract_features(test['image'])

100%|██████████████████████████████████████████████████████████| 7066/7066 [00:26<00:00, 270.64it/s]


In [13]:
x_train = train_features/255.0
x_test = test_features/255.0

In [46]:
pip install -U scikit-learn

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [14]:
from sklearn.preprocessing import LabelEncoder

In [15]:
le = LabelEncoder()
le.fit(train['label'])

LabelEncoder()

In [16]:
y_train = le.transform(train['label'])
y_test = le.transform(test['label'])

In [17]:
y_train = to_categorical(y_train, num_classes=7)
y_test = to_categorical(y_test, num_classes=7)

In [18]:
model = Sequential()

# convolutional layers
model.add(Conv2D(128, kernel_size=(3,3), activation='relu', input_shape=(48, 48, 1)))
model.add(MaxPooling2D(pool_size=(2,2)))
model.add(Dropout(0.4))

model.add(Conv2D(256, kernel_size=(3,3), activation='relu'))
model.add(MaxPooling2D(pool_size=(2,2)))
model.add(Dropout(0.4))

model.add(Conv2D(512, kernel_size=(3,3), activation='relu'))
model.add(MaxPooling2D(pool_size=(2,2)))
model.add(Dropout(0.4))

model.add(Conv2D(512, kernel_size=(3,3), activation='relu'))
model.add(MaxPooling2D(pool_size=(2,2)))
model.add(Dropout(0.4))

model.add(Flatten())

# fully connected layers
model.add(Dense(512, activation='relu'))
model.add(Dropout(0.4))

model.add(Dense(256, activation='relu'))
model.add(Dropout(0.3))

# output layer
model.add(Dense(7, activation='softmax'))

c:\student-project\emotion-detection\.venv\Lib\site-packages\keras\src\layers\convolutional\base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [22]:
model.compile(optimizer = 'adam', loss = 'categorical_crossentropy', metrics = 'accuracy')

ValueError: Expected `metrics` argument to be a list, tuple, or dict. Received instead: metrics=accuracy of type <class 'str'>

model.fit(x= x_train, y= y_train, batch_size = 128, epochs = 100, validation_data = (x_test, y_test))

In [54]:
print(x_train.shape)
print(y_train.shape)
print(x_test.shape)
print(y_test.shape)

(28821, 48, 48)
(28821, 7)
(7066, 48, 48)
(7066, 7)


In [19]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                      │ (None, 46, 46, 128)         │           1,280 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling2d (MaxPooling2D)         │ (None, 23, 23, 128)         │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout (Dropout)                    │ (None, 23, 23, 128)         │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv2d_1 (Conv2D)                    │ (None, 21, 21, 256)         │         295,168 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling2d_1 (MaxPooling2D)       │ (None, 10, 10, 256)         │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout_1 (Dropout)                  │ (None, 10, 10, 256)         │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv2d_2 (Conv2D)                    │ (None, 8, 8, 512)           │       1,180,160 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling2d_2 (MaxPooling2D)       │ (None, 4, 4, 512)           │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout_2 (Dropout)                  │ (None, 4, 4, 512)           │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv2d_3 (Conv2D)                    │ (None, 2, 2, 512)           │       2,359,808 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling2d_3 (MaxPooling2D)       │ (None, 1, 1, 512)           │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout_3 (Dropout)                  │ (None, 1, 1, 512)           │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ flatten (Flatten)                    │ (None, 512)                 │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense (Dense)                        │ (None, 512)                 │         262,656 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout_4 (Dropout)                  │ (None, 512)                 │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_1 (Dense)                      │ (None, 256)                 │         131,328 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout_5 (Dropout)                  │ (None, 256)                 │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_2 (Dense)                      │ (None, 7)                   │           1,799 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 4,232,199 (16.14 MB)

 Trainable params: 4,232,199 (16.14 MB)

 Non-trainable params: 0 (0.00 B)

In [20]:
model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

In [21]:
history = model.fit(
    x_train[:100],
    y_train[:100],
    batch_size=32,
    epochs=1
)

4/4 ━━━━━━━━━━━━━━━━━━━━ 23s 913ms/step - accuracy: 0.7100 - loss: 0.8147


In [23]:
history = model.fit(
    x_train,
    y_train,
    batch_size=128,
    epochs=100,
    validation_data=(x_test, y_test)
)

Epoch 1/100
226/226 ━━━━━━━━━━━━━━━━━━━━ 879s 4s/step - accuracy: 0.2451 - loss: 2.3410 - val_accuracy: 0.2583 - val_loss: 1.8101
Epoch 2/100
226/226 ━━━━━━━━━━━━━━━━━━━━ 1130s 5s/step - accuracy: 0.2513 - loss: 1.8088 - val_accuracy: 0.2652 - val_loss: 1.7902
Epoch 3/100
226/226 ━━━━━━━━━━━━━━━━━━━━ 802s 4s/step - accuracy: 0.2542 - loss: 1.7911 - val_accuracy: 0.2751 - val_loss: 1.7489
Epoch 4/100
226/226 ━━━━━━━━━━━━━━━━━━━━ 746s 3s/step - accuracy: 0.2878 - loss: 1.7325 - val_accuracy: 0.3237 - val_loss: 1.6743
Epoch 5/100
226/226 ━━━━━━━━━━━━━━━━━━━━ 755s 3s/step - accuracy: 0.3076 - loss: 1.6949 - val_accuracy: 0.3431 - val_loss: 1.6445
Epoch 6/100
226/226 ━━━━━━━━━━━━━━━━━━━━ 889s 4s/step - accuracy: 0.3361 - loss: 1.6512 - val_accuracy: 0.3807 - val_loss: 1.5775
Epoch 7/100
226/226 ━━━━━━━━━━━━━━━━━━━━ 821s 4s/step - accuracy: 0.3684 - loss: 1.5966 - val_accuracy: 0.4222 - val_loss: 1.4823
Epoch 8/100
226/226 ━━━━━━━━━━━━━━━━━━━━ 976s 4s/step - accuracy: 0.3992 - loss: 1.5391 -

KeyboardInterrupt: 